# Notebook 5: Human labels vs the dataset labels (RQ3)

**Before running:** a GPU helps but isn't required (only 150 pairs). Runtime → Change runtime type → **T4 GPU** if one is free. Run time about 5 minutes.

**Where the human labels are read from:** `MyDrive/resume-job-matching/data/human_labelling_sample.xlsx`.
If you labelled it in **Google Sheets** (not Excel), Drive keeps the original .xlsx unchanged. In that case: File → Download → Microsoft Excel (.xlsx), then upload it back to the same folder with the same name (replace).

**What this notebook answers**
1. **How much do you agree with the dataset?** Raw agreement and Cohen's κ (plain and ordinal-weighted), with confidence intervals. We also split it into the two decisions: *fit vs no fit* and *good vs potential*.
2. **Is the Good/Potential label a resume-level artifact?** Notebook 3b showed that a *resume-prior* baseline (which never reads the job) separates Good from Potential with AUC 0.727, while every content method is at chance. If your job-specific labels are the better ground truth, the resume-prior should **drop** on the human labels, and content methods (E5, the LLM) should **rise**.
3. **Which methods agree with a human?** Every method re-scored on the human labels, including the Qwen3 LLM judge (if Notebook 4 has been run) with macro-F1, comparable with Jin (2025).

**Partial labels:** you can run this before you finish. It uses the labelled rows and prints a warning. The numbers only count for the paper once all 150 are done.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers

import os, gc
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score, roc_auc_score, f1_score, confusion_matrix

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

LABELS = ['No Fit', 'Potential Fit', 'Good Fit']        # ordinal order: 0, 1, 2
TO_INT = {l: i for i, l in enumerate(LABELS)}
ALIASES = {'good': 'Good Fit', 'good fit': 'Good Fit', 'potential': 'Potential Fit', 'potential fit': 'Potential Fit',
           'no': 'No Fit', 'no fit': 'No Fit', 'none': 'No Fit'}

sheet = pd.read_excel(f'{DATA}/human_labelling_sample.xlsx')
key = pd.read_csv(f'{DATA}/human_labelling_key.csv')
sheet['human'] = sheet.your_label.fillna('').astype(str).str.strip().str.lower().map(ALIASES)
bad = sheet[sheet.your_label.notna() & (sheet.your_label.astype(str).str.strip() != '') & sheet.human.isna()]
if len(bad):
    print('Could not read these labels, fix them in the sheet:'); print(bad[['pair_no', 'your_label']])

lab = sheet[sheet.human.notna()].merge(key, on='pair_no', how='left')
assert lab.label.notna().all(), 'pair_no mismatch between the sheet and the key file'
lab = lab.rename(columns={'label': 'dataset'})
lab['h'] = lab.human.map(TO_INT); lab['d'] = lab.dataset.map(TO_INT)
lab['pair_key'] = lab.job_id + '|' + lab.resume_id

FINAL = len(lab) == len(sheet)
print(f'{len(lab)} of {len(sheet)} pairs labelled.' + ('' if FINAL else '  ⚠ PARTIAL: preview only, not for the paper.'))

## 1. Agreement with the dataset labels

In [ ]:
def kappa_ci(a, b, weights=None, n=2000):
    rng = np.random.default_rng(SEED); a, b = np.asarray(a), np.asarray(b); ks = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        if len(set(a[i])) > 1 or len(set(b[i])) > 1:
            ks.append(cohen_kappa_score(a[i], b[i], weights=weights))
    return np.percentile(ks, [2.5, 97.5])

def agreement(a, b, name):
    k = cohen_kappa_score(a, b); lo, hi = kappa_ci(a, b)
    row = {'decision': name, 'pairs': len(a), 'raw agreement': round(np.mean(np.asarray(a) == np.asarray(b)), 3),
           'kappa': round(k, 3), 'kappa CI': f'[{lo:.3f}, {hi:.3f}]'}
    if len(set(a) | set(b)) > 2:
        kw = cohen_kappa_score(a, b, weights='linear'); lo, hi = kappa_ci(a, b, 'linear')
        row.update({'weighted kappa': round(kw, 3), 'weighted kappa CI': f'[{lo:.3f}, {hi:.3f}]'})
    return row

both_fit = lab[(lab.h > 0) & (lab.d > 0)]
agree = pd.DataFrame([
    agreement(lab.h, lab.d, '3 labels'),
    agreement((lab.h > 0).astype(int), (lab.d > 0).astype(int), 'fit vs no fit'),
    agreement((both_fit.h == 2).astype(int), (both_fit.d == 2).astype(int), 'good vs potential (both say fit)'),
])
display(agree)

cm = pd.DataFrame(confusion_matrix(lab.dataset, lab.human, labels=LABELS),
                  index=['dataset: ' + l for l in LABELS], columns=['you: ' + l for l in LABELS])
display(cm)
print('Human label counts:', lab.human.value_counts().reindex(LABELS).to_dict())
print('Dataset label counts:', lab.dataset.value_counts().reindex(LABELS).to_dict())
print('Landis & Koch: <0.20 slight, 0.21-0.40 fair, 0.41-0.60 moderate, 0.61-0.80 substantial, >0.80 almost perfect')

## 2. Method scores for the labelled pairs
The same settings as Notebook 3b: TF-IDF fitted on all texts (no labels), zero-shot E5 / BGE / MPNet, and the **resume-prior**, which scores a resume by its mean *dataset* label with every **other** job and never reads this job.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet') for s in ('train', 'val', 'test')], ignore_index=True)
texts = all_df.drop_duplicates('job_id').set_index('job_id').job_text
rtexts = all_df.drop_duplicates('resume_id').set_index('resume_id').resume_text
lab['job_text'] = lab.job_id.map(texts); lab['resume_text'] = lab.resume_id.map(rtexts)

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tfidf.fit(list(texts) + list(rtexts))
J, R = tfidf.transform(lab.job_text), tfidf.transform(lab.resume_text)
lab['TF-IDF'] = np.asarray(J.multiply(R).sum(axis=1)).ravel()   # rows are L2-normalised, so this is cosine

for name, hf_id, qp, dp in [('E5-base', 'intfloat/e5-base-v2', 'query: ', 'passage: '),
                            ('BGE-base', 'BAAI/bge-base-en-v1.5', '', ''),
                            ('MPNet-base', 'sentence-transformers/all-mpnet-base-v2', '', '')]:
    m = SentenceTransformer(hf_id, device=device); m.max_seq_length = 512
    je = m.encode([qp + t for t in lab.job_text], normalize_embeddings=True, batch_size=16)
    re_ = m.encode([dp + t for t in lab.resume_text], normalize_embeddings=True, batch_size=16)
    lab[name] = (je * re_).sum(axis=1)
    del m; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

tot = all_df.groupby('resume_id').relevance.agg(['sum', 'count'])
own = all_df.set_index(['job_id', 'resume_id']).relevance
def prior(jid, rid):
    s, c = tot.at[rid, 'sum'] - own.at[(jid, rid)], tot.at[rid, 'count'] - 1
    return s / c if c > 0 else all_df.relevance.mean()
lab['Resume-prior'] = [prior(j, r) for j, r in zip(lab.job_id, lab.resume_id)]

METHODS = ['Resume-prior', 'TF-IDF', 'E5-base', 'BGE-base', 'MPNet-base']
llm_path = f'{RESULTS}/04_llm_pair_scores.parquet'
if os.path.exists(llm_path):
    llm = pd.read_parquet(llm_path).set_index('pair_key')
    for col, name in [('qwen4b', 'Qwen3-4B judge'), ('qwen1_7b', 'Qwen3-1.7B judge')]:
        if col in llm:
            lab[name] = lab.pair_key.map(llm[col]); lab[name + ' label'] = lab.pair_key.map(llm[col + '_label'])
            METHODS.append(name)
    print('LLM scores found for', int(lab['Qwen3-4B judge'].notna().sum()), 'of', len(lab), 'pairs '
          '(Notebook 4 scored only jobs with a usable ranking list)')
else:
    print('Notebook 4 results not found yet: the LLM rows are skipped. Run this again after Notebook 4.')

## 3. The key test: which labels does each method explain?
Pooled pair-level AUC (0.5 = chance) with 95% bootstrap intervals, against **the dataset label** and against **the human label**.
- **fit vs no**: all pairs.
- **good vs potential**: only pairs that the given labeller called a fit.

**What would support the paper's claim:** the resume-prior is high on the dataset's good-vs-potential but falls toward 0.5 on yours, while the content methods do the opposite.

In [ ]:
def auc_ci(y, s, n=2000):
    y, s = np.asarray(y), np.asarray(s, dtype=float); ok = ~np.isnan(s); y, s = y[ok], s[ok]
    if len(set(y)) < 2: return np.nan, np.nan, np.nan, len(y)
    rng = np.random.default_rng(SEED); boots = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: boots.append(roc_auc_score(y[i], s[i]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return roc_auc_score(y, s), lo, hi, len(y)

rows = []
for m in METHODS:
    for who, col in [('dataset', 'd'), ('you', 'h')]:
        fit = lab[lab[col] > 0]
        for task, y, s in [('fit vs no', (lab[col] > 0).astype(int), lab[m]),
                           ('good vs potential', (fit[col] == 2).astype(int), fit[m])]:
            a, lo, hi, n = auc_ci(y, s)
            rows.append({'method': m, 'task': task, 'labels': who, 'AUC': round(a, 3), 'CI': f'[{lo:.3f}, {hi:.3f}]', 'pairs': n})
auc = pd.DataFrame(rows)
wide = auc.pivot_table(index='method', columns=['task', 'labels'], values='AUC', aggfunc='first').reindex(METHODS)
display(wide)

### 3b. Is the change significant?
For each method, the drop or gain in good-vs-potential AUC when the human labels replace the dataset labels. The pairs differ between the two label sets, so we bootstrap over all labelled pairs and recompute both AUCs in each resample.

In [ ]:
def shift_test(m, n=2000):
    rng = np.random.default_rng(SEED); diffs = []
    def gp(df, col):
        f = df[(df[col] > 0) & df[m].notna()]; y = (f[col] == 2).astype(int)
        return roc_auc_score(y, f[m]) if 0 < y.sum() < len(y) else np.nan
    base = gp(lab, 'h') - gp(lab, 'd')
    for _ in range(n):
        b = lab.iloc[rng.integers(0, len(lab), len(lab))]
        diffs.append(gp(b, 'h') - gp(b, 'd'))
    diffs = np.asarray(diffs); diffs = diffs[~np.isnan(diffs)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return {'method': m, 'AUC(you) - AUC(dataset)': round(base, 3), 'CI': f'[{lo:.3f}, {hi:.3f}]', 'p (two-sided)': round(min(p, 1), 4)}

shift = pd.DataFrame([shift_test(m) for m in METHODS])
display(shift)

## 4. LLM judge as a classifier: macro-F1 against the dataset and against you

In [ ]:
f1_rows = []
for name in [m for m in METHODS if m.endswith('judge')]:
    d = lab[lab[name + ' label'].notna()]
    for who, col in [('dataset', 'dataset'), ('you', 'human')]:
        f1_rows.append({'model': name, 'labels': who, 'pairs': len(d),
                        'macro-F1': round(f1_score(d[col], d[name + ' label'], average='macro', labels=LABELS), 3),
                        'kappa': round(cohen_kappa_score(d[col], d[name + ' label']), 3)})
f1_tab = pd.DataFrame(f1_rows)
display(f1_tab if len(f1_tab) else 'No LLM scores yet.')
print('Reference: Jin (2025), macro-F1 0.450 against the dataset labels on the full test set.')

## 5. Strongest disagreements, for the paper's qualitative analysis
Pairs where one side said **Good Fit** and the other said **No Fit**. Read two or three of these again and note *why*. Short, concrete reasons (e.g. "same field, but the job needs 8 years and the resume is a graduate") make the discussion section convincing.

In [ ]:
strong = lab[(lab.h - lab.d).abs() == 2][['pair_no', 'dataset', 'human', 'note (optional)', 'Resume-prior', 'E5-base']]
print(len(strong), 'strong disagreements')
display(strong.sort_values('pair_no'))

## 6. Save

In [ ]:
tag = '' if FINAL else '_PARTIAL'
agree.to_csv(f'{RESULTS}/05_agreement{tag}.csv', index=False)
cm.to_csv(f'{RESULTS}/05_confusion{tag}.csv')
auc.to_csv(f'{RESULTS}/05_auc_by_labels{tag}.csv', index=False)
shift.to_csv(f'{RESULTS}/05_auc_shift{tag}.csv', index=False)
if len(f1_tab): f1_tab.to_csv(f'{RESULTS}/05_llm_f1{tag}.csv', index=False)
lab.drop(columns=['job_description', 'resume', 'job_text', 'resume_text']).to_csv(f'{RESULTS}/05_labelled_pairs{tag}.csv', index=False)
print('saved' + (' (PARTIAL files)' if tag else ''))